# 🚀 Real-Time Crypto Price Trend Prediction

## Hệ thống dự báo xu hướng giá Crypto theo thời gian thực

**Pipeline:**
```
Binance API → Producer → Kafka Queue (Giả lập) → Spark Streaming → Feature Engineering (RSI/MACD) → Prediction → Storage (CSV) → Visualization
```

### Các thành phần:
| Thành phần | Giải pháp |
|---|---|
| Data Source | Binance REST API (klines endpoint) |
| Message Queue | Python `queue.Queue` (giả lập Kafka) |
| Stream Processing | PySpark Structured Streaming |
| Feature Engineering | RSI, MACD (tính thủ công + pandas) |
| Prediction | Rule-based + Logistic Regression (scikit-learn) |
| Storage | CSV file (giả lập Cassandra/HBase) |
| Visualization | Matplotlib + IPython display |

> **Lưu ý:** Kafka thật rất khó chạy trên Colab do giới hạn tài nguyên và Java. Notebook này dùng `queue.Queue` + threading để giả lập Kafka producer/consumer với đầy đủ logic tương đương.

## Bước 1: Cài đặt môi trường

Cài đặt PySpark và các thư viện cần thiết.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Bước 1: Cài đặt môi trường
# Cài PySpark, pandas, matplotlib, scikit-learn và requests
# ─────────────────────────────────────────────────────────────
!pip install pyspark==3.5.0 requests pandas matplotlib scikit-learn --quiet

import sys
print(f"Python version: {sys.version}")

import pyspark
print(f"PySpark version: {pyspark.__version__}")

import pandas as pd
import sklearn
print(f"Pandas version: {pd.__version__}")
print(f"scikit-learn version: {sklearn.__version__}")
print("\n✅ Môi trường đã sẵn sàng!")

## Bước 2: Import thư viện và cấu hình

Import tất cả thư viện cần thiết và thiết lập các hằng số cấu hình.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Bước 2: Import thư viện và cấu hình
# ─────────────────────────────────────────────────────────────
import os
import json
import time
import queue
import threading
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
from collections import deque
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

# scikit-learn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# PySpark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType,
    DoubleType, LongType, TimestampType
)

# ─── Cấu hình ────────────────────────────────────────────────
SYMBOL          = "BTCUSDT"       # Cặp tiền tệ
INTERVAL        = "1m"            # Khung thời gian: 1 phút
FETCH_LIMIT     = 200             # Số nến lịch sử ban đầu
STREAM_INTERVAL = 5               # Giây giữa mỗi lần fetch mới
STREAM_ROUNDS   = 20              # Số lần lấy dữ liệu streaming
RSI_PERIOD      = 14              # Chu kỳ tính RSI
MACD_FAST       = 12              # EMA nhanh cho MACD
MACD_SLOW       = 26              # EMA chậm cho MACD
MACD_SIGNAL     = 9               # Đường tín hiệu MACD
OUTPUT_CSV      = "/content/crypto_predictions.csv"  # File lưu kết quả

# Binance API endpoint
BINANCE_URL = "https://api.binance.com/api/v3/klines"

print("✅ Cấu hình:")
print(f"   Symbol        : {SYMBOL}")
print(f"   Interval      : {INTERVAL}")
print(f"   RSI Period    : {RSI_PERIOD}")
print(f"   MACD          : ({MACD_FAST}, {MACD_SLOW}, {MACD_SIGNAL})")
print(f"   Stream rounds : {STREAM_ROUNDS} x {STREAM_INTERVAL}s")
print(f"   Output CSV    : {OUTPUT_CSV}")

## Bước 3: Lấy dữ liệu từ Binance API

Viết hàm fetch dữ liệu OHLCV (Open/High/Low/Close/Volume) từ Binance REST API.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Bước 3: Lấy dữ liệu từ Binance API
# Binance trả về mảng klines với các trường:
#  [0]  open_time, [1] open, [2] high, [3] low,
#  [4]  close,     [5] volume, [6] close_time, ...
# ─────────────────────────────────────────────────────────────

def fetch_klines(symbol: str, interval: str, limit: int = 100) -> pd.DataFrame:
    """Lấy dữ liệu nến (OHLCV) từ Binance API.

    Args:
        symbol:   Cặp giao dịch, ví dụ 'BTCUSDT'.
        interval: Khung thời gian ('1m', '5m', '1h', ...).
        limit:    Số nến cần lấy (tối đa 1000).

    Returns:
        DataFrame với các cột: timestamp, open, high, low, close, volume.
    """
    params = {"symbol": symbol, "interval": interval, "limit": limit}
    try:
        resp = requests.get(BINANCE_URL, params=params, timeout=10)
        resp.raise_for_status()
        raw = resp.json()
    except requests.exceptions.RequestException as exc:
        print(f"⚠️  Không thể kết nối Binance API: {exc}")
        print("   → Dùng dữ liệu giả lập thay thế.")
        return _generate_mock_data(limit)

    records = []
    for k in raw:
        records.append({
            "timestamp": datetime.utcfromtimestamp(k[0] / 1000),
            "open":      float(k[1]),
            "high":      float(k[2]),
            "low":       float(k[3]),
            "close":     float(k[4]),
            "volume":    float(k[5]),
        })

    df = pd.DataFrame(records)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    return df


def _generate_mock_data(n: int = 200) -> pd.DataFrame:
    """Tạo dữ liệu nến giả lập khi không có kết nối API."""
    np.random.seed(42)
    base_price = 65_000.0
    prices = [base_price]
    for _ in range(n - 1):
        change = np.random.normal(0, 200)
        prices.append(max(prices[-1] + change, 1000))

    now = datetime.utcnow()
    timestamps = [now - timedelta(minutes=n - i) for i in range(n)]

    records = []
    for i, (ts, p) in enumerate(zip(timestamps, prices)):
        noise = abs(np.random.normal(0, 100))
        records.append({
            "timestamp": ts,
            "open":      round(p - np.random.uniform(0, noise), 2),
            "high":      round(p + np.random.uniform(0, noise * 1.5), 2),
            "low":       round(p - np.random.uniform(0, noise * 1.5), 2),
            "close":     round(p, 2),
            "volume":    round(np.random.uniform(10, 500), 4),
        })

    df = pd.DataFrame(records)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    return df


# ─── Test fetch ──────────────────────────────────────────────
print(f"📡 Đang lấy {FETCH_LIMIT} nến {INTERVAL} cho {SYMBOL}...")
df_history = fetch_klines(SYMBOL, INTERVAL, FETCH_LIMIT)
print(f"✅ Lấy thành công {len(df_history)} nến.")
print(f"   Thời gian: {df_history['timestamp'].min()} → {df_history['timestamp'].max()}")
print(f"   Giá đóng cửa mới nhất: ${df_history['close'].iloc[-1]:,.2f}")
df_history.tail()

## Bước 4: Kafka Queue Giả lập (Producer/Consumer)

Vì Kafka thật rất khó chạy trên Colab, ta dùng `queue.Queue` của Python để giả lập:
- **Producer**: Thread liên tục fetch giá mới từ API và đẩy vào queue
- **Consumer**: Đọc message từ queue và xử lý

In [ ]:
# ─────────────────────────────────────────────────────────────
# Bước 4: Kafka Queue Giả lập
# Dùng queue.Queue + threading thay vì Kafka thật.
# Giao diện tương tự KafkaProducer/KafkaConsumer:
#   producer.send(topic, value=message)
#   consumer.poll() → trả về message
# ─────────────────────────────────────────────────────────────

class MockKafkaBroker:
    """Giả lập Kafka broker trong bộ nhớ với nhiều topic."""

    def __init__(self):
        self._topics: dict[str, queue.Queue] = {}
        self._lock = threading.Lock()

    def _get_topic(self, topic: str) -> queue.Queue:
        with self._lock:
            if topic not in self._topics:
                self._topics[topic] = queue.Queue(maxsize=1000)
            return self._topics[topic]

    def send(self, topic: str, value: dict) -> None:
        """Đẩy message vào topic (giống KafkaProducer.send)."""
        q = self._get_topic(topic)
        try:
            q.put_nowait(value)
        except queue.Full:
            # Bỏ message cũ nhất nếu queue đầy
            try:
                q.get_nowait()
            except queue.Empty:
                pass
            q.put_nowait(value)

    def poll(self, topic: str, timeout: float = 1.0) -> list[dict]:
        """Lấy tất cả message có sẵn trong topic (giống KafkaConsumer.poll)."""
        q = self._get_topic(topic)
        messages = []
        deadline = time.time() + timeout
        while time.time() < deadline:
            try:
                msg = q.get_nowait()
                messages.append(msg)
            except queue.Empty:
                break
        return messages

    def qsize(self, topic: str) -> int:
        return self._get_topic(topic).qsize()


# ─── Producer: fetch dữ liệu và đẩy vào Kafka queue ──────────
class CryptoProducer:
    """Fetch giá crypto từ API và đẩy vào Kafka (giả lập)."""

    TOPIC = "crypto-prices"

    def __init__(self, broker: MockKafkaBroker, symbol: str, interval: str):
        self.broker   = broker
        self.symbol   = symbol
        self.interval = interval
        self._stop    = threading.Event()
        self._thread  = threading.Thread(target=self._run, daemon=True)
        self._last_ts = None   # Tránh gửi nến trùng

    def start(self) -> None:
        self._thread.start()
        print(f"✅ Producer started → topic='{self.TOPIC}'")

    def stop(self) -> None:
        self._stop.set()
        self._thread.join(timeout=5)
        print("🛑 Producer stopped.")

    def _run(self) -> None:
        while not self._stop.is_set():
            try:
                df = fetch_klines(self.symbol, self.interval, limit=5)
                for _, row in df.iterrows():
                    ts = row["timestamp"]
                    if self._last_ts is not None and ts <= self._last_ts:
                        continue
                    self._last_ts = ts
                    message = {
                        "symbol":    self.symbol,
                        "timestamp": ts.isoformat(),
                        "open":      row["open"],
                        "high":      row["high"],
                        "low":       row["low"],
                        "close":     row["close"],
                        "volume":    row["volume"],
                    }
                    self.broker.send(self.TOPIC, message)
            except Exception as exc:
                print(f"⚠️  Producer error: {exc}")
            self._stop.wait(timeout=STREAM_INTERVAL)


# ─── Khởi tạo broker ──────────────────────────────────────────
kafka_broker = MockKafkaBroker()
producer     = CryptoProducer(kafka_broker, SYMBOL, INTERVAL)
print("✅ Kafka broker (giả lập) và Producer đã được khởi tạo.")
print(f"   Topic: '{CryptoProducer.TOPIC}'")

## Bước 5: Tính RSI và MACD

Tính các chỉ báo kỹ thuật:
- **RSI (Relative Strength Index)**: Đo sức mạnh xu hướng giá, giá trị 0-100
- **MACD (Moving Average Convergence Divergence)**: Chênh lệch giữa EMA nhanh và chậm

In [ ]:
# ─────────────────────────────────────────────────────────────
# Bước 5: Tính RSI và MACD
# ─────────────────────────────────────────────────────────────

def compute_rsi(prices: pd.Series, period: int = 14) -> pd.Series:
    """Tính RSI (Relative Strength Index).

    Công thức:
        delta   = price.diff()
        gain    = EWM(delta khi > 0, span=period)
        loss    = EWM(|delta| khi < 0, span=period)
        RS      = gain / loss
        RSI     = 100 - 100 / (1 + RS)

    Args:
        prices: Series giá đóng cửa.
        period: Chu kỳ tính (mặc định 14).

    Returns:
        Series RSI (0-100), NaN với các giá trị đầu không đủ dữ liệu.
    """
    delta = prices.diff()
    gain  = delta.clip(lower=0)
    loss  = (-delta).clip(lower=0)

    avg_gain = gain.ewm(span=period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(span=period, min_periods=period, adjust=False).mean()

    rs  = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    return rsi


def compute_macd(
    prices:  pd.Series,
    fast:    int = 12,
    slow:    int = 26,
    signal:  int = 9,
) -> pd.DataFrame:
    """Tính MACD, Signal line và Histogram.

    Công thức:
        MACD_line   = EMA(fast) - EMA(slow)
        Signal_line = EMA(MACD_line, span=signal)
        Histogram   = MACD_line - Signal_line

    Args:
        prices: Series giá đóng cửa.
        fast:   Chu kỳ EMA nhanh (mặc định 12).
        slow:   Chu kỳ EMA chậm (mặc định 26).
        signal: Chu kỳ đường tín hiệu (mặc định 9).

    Returns:
        DataFrame với cột: macd, macd_signal, macd_hist.
    """
    ema_fast    = prices.ewm(span=fast,   adjust=False).mean()
    ema_slow    = prices.ewm(span=slow,   adjust=False).mean()
    macd_line   = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    histogram   = macd_line - signal_line

    return pd.DataFrame({
        "macd":        macd_line,
        "macd_signal": signal_line,
        "macd_hist":   histogram,
    })


def add_indicators(df: pd.DataFrame) -> pd.DataFrame:
    """Thêm RSI, MACD và các feature phụ trợ vào DataFrame.

    Args:
        df: DataFrame với cột 'close'.

    Returns:
        DataFrame đã bổ sung thêm cột chỉ báo.
    """
    df = df.copy()
    df["rsi"]        = compute_rsi(df["close"], RSI_PERIOD)
    macd_df          = compute_macd(df["close"], MACD_FAST, MACD_SLOW, MACD_SIGNAL)
    df["macd"]       = macd_df["macd"]
    df["macd_signal"]= macd_df["macd_signal"]
    df["macd_hist"]  = macd_df["macd_hist"]

    # Feature thêm: chênh lệch giá và MA
    df["price_change"]  = df["close"].pct_change() * 100
    df["ma_20"]         = df["close"].rolling(20).mean()
    df["ma_50"]         = df["close"].rolling(50).mean()
    df["volatility"]    = df["close"].rolling(20).std()
    return df


# ─── Test với dữ liệu lịch sử ────────────────────────────────
df_with_indicators = add_indicators(df_history)
valid_rows = df_with_indicators.dropna(subset=["rsi", "macd"])
print(f"✅ Đã tính chỉ báo cho {len(valid_rows)} nến (từ {len(df_with_indicators)} tổng).")
print("\nMẫu dữ liệu 5 nến mới nhất:")
cols_show = ["timestamp", "close", "rsi", "macd", "macd_signal", "macd_hist"]
display(valid_rows[cols_show].tail())

## Bước 6: Logic dự đoán xu hướng (Rule-based)

Áp dụng quy tắc kỹ thuật:
- **RSI > 70** → **SELL** (quá mua)
- **RSI < 30** → **BUY** (quá bán)
- **MACD cắt lên** (histogram > 0 và trước đó ≤ 0) → **UP**
- **MACD cắt xuống** (histogram < 0 và trước đó ≥ 0) → **DOWN**

In [ ]:
# ─────────────────────────────────────────────────────────────
# Bước 6: Logic dự đoán xu hướng (Rule-based)
# ─────────────────────────────────────────────────────────────

def rule_based_prediction(df: pd.DataFrame) -> pd.DataFrame:
    """Gán nhãn dự đoán dựa trên RSI và MACD.

    Quy tắc (theo thứ tự ưu tiên):
        1. RSI > 70                         → SELL
        2. RSI < 30                         → BUY
        3. MACD_hist > 0 và trước đó <= 0   → UP  (MACD cắt lên)
        4. MACD_hist < 0 và trước đó >= 0   → DOWN (MACD cắt xuống)
        5. Mặc định theo chiều MACD_hist     → UP / DOWN

    Args:
        df: DataFrame có cột rsi, macd_hist.

    Returns:
        DataFrame với cột 'signal' và 'prediction' (UP/DOWN/BUY/SELL).
    """
    df = df.copy()
    prev_hist   = df["macd_hist"].shift(1)

    macd_cross_up   = (df["macd_hist"] > 0) & (prev_hist <= 0)
    macd_cross_down = (df["macd_hist"] < 0) & (prev_hist >= 0)

    conditions = [
        df["rsi"] > 70,
        df["rsi"] < 30,
        macd_cross_up,
        macd_cross_down,
        df["macd_hist"] >= 0,
    ]
    choices = ["SELL", "BUY", "UP", "DOWN", "UP"]
    df["signal"]     = np.select(conditions, choices, default="DOWN")
    df["prediction"] = df["signal"]  # Alias để hiển thị
    return df


# ─── Test ────────────────────────────────────────────────────
df_predicted = rule_based_prediction(df_with_indicators)
valid = df_predicted.dropna(subset=["rsi", "macd"])

print("✅ Phân phối tín hiệu dự đoán:")
print(valid["signal"].value_counts().to_string())
print()

cols = ["timestamp", "close", "rsi", "macd", "macd_hist", "signal"]
print("Mẫu 10 nến mới nhất:")
display(valid[cols].tail(10))

## Bước 7: Mô hình ML — Logistic Regression & Random Forest

Huấn luyện mô hình ML sử dụng dữ liệu lịch sử để dự đoán xu hướng tăng/giảm (1/0).

In [ ]:
# ─────────────────────────────────────────────────────────────
# Bước 7: Mô hình ML — Logistic Regression & Random Forest
# ─────────────────────────────────────────────────────────────

FEATURE_COLS = ["rsi", "macd", "macd_signal", "macd_hist",
                "price_change", "volatility"]

def prepare_ml_data(df: pd.DataFrame) -> tuple:
    """Chuẩn bị X, y cho ML.

    Nhãn y = 1 (UP) nếu giá đóng cửa nến tiếp theo > hiện tại, else 0 (DOWN).

    Returns:
        Tuple (X_train, X_test, y_train, y_test, scaler)
    """
    df = df.copy()
    # Tạo nhãn thực (future price direction)
    df["future_close"] = df["close"].shift(-1)
    df["label"]        = (df["future_close"] > df["close"]).astype(int)

    clean = df[FEATURE_COLS + ["label"]].dropna()
    X = clean[FEATURE_COLS].values
    y = clean["label"].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, shuffle=False
    )
    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)
    return X_train, X_test, y_train, y_test, scaler


def train_models(df: pd.DataFrame) -> dict:
    """Huấn luyện Logistic Regression và Random Forest.

    Returns:
        Dict chứa scaler và các model đã huấn luyện.
    """
    X_train, X_test, y_train, y_test, scaler = prepare_ml_data(df)

    # ── Logistic Regression ──────────────────────────────────
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(X_train, y_train)
    lr_acc = lr.score(X_test, y_test)

    # ── Random Forest ────────────────────────────────────────
    rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_acc = rf.score(X_test, y_test)

    print("\n📊 Kết quả Logistic Regression:")
    print(f"   Accuracy: {lr_acc:.4f}")
    print(classification_report(y_test, lr.predict(X_test),
                                 target_names=["DOWN", "UP"]))

    print("\n📊 Kết quả Random Forest:")
    print(f"   Accuracy: {rf_acc:.4f}")
    print(classification_report(y_test, rf.predict(X_test),
                                 target_names=["DOWN", "UP"]))

    # Feature importance
    fi = dict(zip(FEATURE_COLS, rf.feature_importances_))
    fi_sorted = sorted(fi.items(), key=lambda x: x[1], reverse=True)
    print("\n🌲 Random Forest Feature Importance:")
    for feat, imp in fi_sorted:
        bar = "█" * int(imp * 40)
        print(f"   {feat:<15} {bar} {imp:.4f}")

    return {"scaler": scaler, "lr": lr, "rf": rf}


def ml_predict(models: dict, features: list) -> dict:
    """Dự đoán cho một nến mới bằng ML models.

    Args:
        models:   Dict từ train_models().
        features: List các giá trị [rsi, macd, macd_signal,
                   macd_hist, price_change, volatility].

    Returns:
        Dict {lr_pred, rf_pred, lr_proba, rf_proba}.
    """
    scaler = models["scaler"]
    X      = scaler.transform([features])
    lr_pred  = models["lr"].predict(X)[0]
    rf_pred  = models["rf"].predict(X)[0]
    lr_proba = models["lr"].predict_proba(X)[0][1]
    rf_proba = models["rf"].predict_proba(X)[0][1]
    return {
        "lr_pred":  "UP" if lr_pred else "DOWN",
        "rf_pred":  "UP" if rf_pred else "DOWN",
        "lr_proba": round(lr_proba, 4),
        "rf_proba": round(rf_proba, 4),
    }


# ─── Huấn luyện với dữ liệu lịch sử ────────────────────────
print("🤖 Huấn luyện mô hình ML với dữ liệu lịch sử...")
ml_models = train_models(df_with_indicators)
print("\n✅ Mô hình đã huấn luyện xong!")

## Bước 8: PySpark — Khởi tạo SparkSession và Schema

Khởi tạo SparkSession và định nghĩa schema cho dữ liệu streaming.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Bước 8: PySpark — Khởi tạo SparkSession
# ─────────────────────────────────────────────────────────────

spark = (
    SparkSession.builder
    .appName("CryptoStreamingPrediction")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.memory", "2g")
    .config("spark.ui.enabled", "false")   # Tắt Spark UI để tiết kiệm tài nguyên Colab
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print(f"✅ SparkSession đã khởi tạo.")
print(f"   Spark version  : {spark.version}")
print(f"   App name       : {spark.sparkContext.appName}")

# ─── Schema cho dữ liệu streaming ────────────────────────────
PRICE_SCHEMA = StructType([
    StructField("symbol",     StringType(),  True),
    StructField("timestamp",  StringType(),  True),
    StructField("open",       DoubleType(),  True),
    StructField("high",       DoubleType(),  True),
    StructField("low",        DoubleType(),  True),
    StructField("close",      DoubleType(),  True),
    StructField("volume",     DoubleType(),  True),
])

RESULT_SCHEMA = StructType([
    StructField("symbol",       StringType(),  True),
    StructField("timestamp",    StringType(),  True),
    StructField("close",        DoubleType(),  True),
    StructField("rsi",          DoubleType(),  True),
    StructField("macd",         DoubleType(),  True),
    StructField("macd_signal",  DoubleType(),  True),
    StructField("macd_hist",    DoubleType(),  True),
    StructField("signal",       StringType(),  True),
    StructField("ml_lr",        StringType(),  True),
    StructField("ml_rf",        StringType(),  True),
    StructField("lr_proba",     DoubleType(),  True),
    StructField("rf_proba",     DoubleType(),  True),
])

print("\n✅ Schema đã định nghĩa.")

## Bước 9: Spark Streaming Processor

Dùng PySpark để xử lý micro-batch: đọc từ Kafka queue, tính toán chỉ báo và dự đoán bằng Spark UDF.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Bước 9: Spark Streaming Processor
# Xử lý micro-batch: đọc từ queue, tính chỉ báo, dự đoán.
# ─────────────────────────────────────────────────────────────

class SparkStreamProcessor:
    """Xử lý dữ liệu streaming với PySpark theo dạng micro-batch.

    Luồng xử lý:
        1. Nhận micro-batch (list of dict) từ Kafka queue.
        2. Append vào sliding window (deque).
        3. Chuyển window → pandas DataFrame.
        4. Tính RSI, MACD với toàn bộ window.
        5. Áp dụng rule-based và ML prediction.
        6. Tạo Spark DataFrame từ kết quả.
        7. Trả về Spark DataFrame chứa các nến mới nhất.
    """

    WINDOW_SIZE = 100   # Số nến giữ trong sliding window

    def __init__(self, spark_session: SparkSession, models: dict):
        self.spark   = spark_session
        self.models  = models
        self._window = deque(maxlen=self.WINDOW_SIZE)  # Sliding window

        # Seed window với dữ liệu lịch sử
        for _, row in df_history.iterrows():
            self._window.append({
                "symbol":    SYMBOL,
                "timestamp": row["timestamp"].isoformat(),
                "open":      row["open"],
                "high":      row["high"],
                "low":       row["low"],
                "close":     row["close"],
                "volume":    row["volume"],
            })
        print(f"   Window seeded với {len(self._window)} nến lịch sử.")

    def process_batch(self, messages: list[dict]) -> "pyspark.sql.DataFrame | None":
        """Xử lý một micro-batch message.

        Args:
            messages: List các dict message từ Kafka queue.

        Returns:
            Spark DataFrame với kết quả tính toán, hoặc None nếu không có dữ liệu.
        """
        if not messages:
            return None

        # Append message mới vào window
        for msg in messages:
            self._window.append(msg)

        # Chuyển window → pandas DataFrame
        window_df = pd.DataFrame(list(self._window))
        window_df["timestamp"] = pd.to_datetime(window_df["timestamp"])
        window_df = window_df.sort_values("timestamp").reset_index(drop=True)

        # Tính chỉ báo kỹ thuật
        window_df = add_indicators(window_df)
        window_df = rule_based_prediction(window_df)

        # Chỉ lấy các nến mới nhất (từ batch hiện tại)
        batch_ts = {m["timestamp"] for m in messages}
        new_rows = window_df[
            window_df["timestamp"].dt.strftime("%Y-%m-%dT%H:%M:%S").isin(
                {t[:19] for t in batch_ts}
            )
        ].dropna(subset=["rsi", "macd"])

        if new_rows.empty:
            # Lấy nến cuối cùng nếu batch_ts không khớp (định dạng ms)
            new_rows = window_df.tail(len(messages)).dropna(subset=["rsi", "macd"])

        if new_rows.empty:
            return None

        # Thêm ML prediction
        result_rows = []
        for _, row in new_rows.iterrows():
            feat_vals = [
                row.get("rsi", 50),
                row.get("macd", 0),
                row.get("macd_signal", 0),
                row.get("macd_hist", 0),
                row.get("price_change", 0),
                row.get("volatility", 0),
            ]
            # Thay NaN bằng 0 cho ML
            feat_vals = [0.0 if (v is None or (isinstance(v, float) and np.isnan(v))) else float(v)
                         for v in feat_vals]
            ml_result = ml_predict(self.models, feat_vals)

            result_rows.append({
                "symbol":      str(row["symbol"]),
                "timestamp":   str(row["timestamp"]),
                "close":       float(row["close"]),
                "rsi":         float(row["rsi"]),
                "macd":        float(row["macd"]),
                "macd_signal": float(row["macd_signal"]),
                "macd_hist":   float(row["macd_hist"]),
                "signal":      str(row["signal"]),
                "ml_lr":       ml_result["lr_pred"],
                "ml_rf":       ml_result["rf_pred"],
                "lr_proba":    ml_result["lr_proba"],
                "rf_proba":    ml_result["rf_proba"],
            })

        # Tạo Spark DataFrame
        spark_df = self.spark.createDataFrame(result_rows, schema=RESULT_SCHEMA)
        return spark_df


# ─── Khởi tạo processor ───────────────────────────────────────
print("⚙️  Khởi tạo Spark Stream Processor...")
processor = SparkStreamProcessor(spark, ml_models)
print("✅ Spark Stream Processor đã sẵn sàng.")

## Bước 10: Lưu kết quả vào CSV (Giả lập Database)

Lưu kết quả dự đoán vào file CSV — giả lập Cassandra/HBase table.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Bước 10: Lưu kết quả vào CSV (Giả lập Database)
# ─────────────────────────────────────────────────────────────

class ResultStorage:
    """Lưu kết quả dự đoán theo định dạng giống Cassandra table.

    Cấu trúc table (CSV):
        symbol | timestamp | close | rsi | macd | macd_signal |
        macd_hist | signal | ml_lr | ml_rf | lr_proba | rf_proba
    """

    COLUMNS = [
        "symbol", "timestamp", "close", "rsi",
        "macd", "macd_signal", "macd_hist",
        "signal", "ml_lr", "ml_rf", "lr_proba", "rf_proba",
    ]

    def __init__(self, path: str):
        self.path    = path
        self._buffer: list[dict] = []
        self._df     = pd.DataFrame(columns=self.COLUMNS)
        self._lock   = threading.Lock()

        # Xóa file cũ nếu có
        if os.path.exists(path):
            os.remove(path)

    def write(self, spark_df) -> None:
        """Ghi Spark DataFrame vào bộ nhớ và CSV."""
        rows = [row.asDict() for row in spark_df.collect()]
        if not rows:
            return
        with self._lock:
            new_df = pd.DataFrame(rows, columns=self.COLUMNS)
            self._df = pd.concat([self._df, new_df], ignore_index=True)
            # Loại bỏ duplicate theo timestamp
            self._df = self._df.drop_duplicates(subset=["timestamp"])
            self._df = self._df.sort_values("timestamp").reset_index(drop=True)
            self._df.to_csv(self.path, index=False)

    def read(self) -> pd.DataFrame:
        """Đọc toàn bộ dữ liệu đã lưu."""
        with self._lock:
            return self._df.copy()

    def tail(self, n: int = 10) -> pd.DataFrame:
        """Đọc n hàng mới nhất."""
        return self.read().tail(n)

    def count(self) -> int:
        """Số hàng đã lưu."""
        return len(self._df)


storage = ResultStorage(OUTPUT_CSV)
print(f"✅ Storage (CSV) đã khởi tạo tại: {OUTPUT_CSV}")

## Bước 11: Hiển thị kết quả real-time

Hàm vẽ biểu đồ real-time gồm:
1. Giá đóng cửa + MA20/MA50
2. RSI với ngưỡng overbought/oversold
3. MACD Histogram
4. Bảng dữ liệu 10 nến mới nhất

In [ ]:
# ─────────────────────────────────────────────────────────────
# Bước 11: Hàm hiển thị real-time
# ─────────────────────────────────────────────────────────────

SIGNAL_COLORS = {
    "UP":   "#00C851",
    "DOWN": "#FF4444",
    "BUY":  "#00C851",
    "SELL": "#FF4444",
}

def render_dashboard(df_result: pd.DataFrame, round_num: int) -> None:
    """Vẽ dashboard 4 panels cho dữ liệu crypto real-time.

    Panels:
        1. Price Chart với MA20, MA50 và tín hiệu mua/bán.
        2. RSI với ngưỡng 30/70.
        3. MACD Histogram.
        4. Bảng 10 nến mới nhất.

    Args:
        df_result: DataFrame kết quả từ storage.
        round_num: Số thứ tự batch hiện tại.
    """
    if df_result.empty or len(df_result) < 5:
        print("⏳ Chờ đủ dữ liệu...")
        return

    df_plot = df_result.copy()
    df_plot["timestamp"] = pd.to_datetime(df_plot["timestamp"])
    df_plot = df_plot.tail(60).reset_index(drop=True)

    fig, axes = plt.subplots(3, 1, figsize=(14, 10),
                              gridspec_kw={"height_ratios": [3, 1.5, 1.5]})
    fig.patch.set_facecolor("#1E1E2E")
    for ax in axes:
        ax.set_facecolor("#2D2D44")
        ax.tick_params(colors="#CCCCCC")
        ax.spines[:].set_color("#444466")

    ts = df_plot["timestamp"]

    # ── Panel 1: Price + MA ──────────────────────────────────
    ax1 = axes[0]
    ax1.plot(ts, df_plot["close"], color="#4FC3F7", linewidth=1.5, label="Close")

    # Tính MA từ dữ liệu đầy đủ trong storage
    full_df = df_result.copy()
    full_df["timestamp"] = pd.to_datetime(full_df["timestamp"])
    full_df["ma_20"] = full_df["close"].rolling(20).mean()
    full_df["ma_50"] = full_df["close"].rolling(50).mean()
    plot_window = full_df.tail(60)

    ax1.plot(plot_window["timestamp"], plot_window["ma_20"],
             color="#FFD54F", linewidth=1, linestyle="--", label="MA20")
    ax1.plot(plot_window["timestamp"], plot_window["ma_50"],
             color="#FF8A65", linewidth=1, linestyle="--", label="MA50")

    # Vẽ tín hiệu mua/bán
    for _, row in df_plot.iterrows():
        if row["signal"] in ("BUY", "UP"):
            ax1.scatter(row["timestamp"], row["close"],
                        marker="^", color="#00E676", s=60, zorder=5)
        elif row["signal"] in ("SELL", "DOWN"):
            ax1.scatter(row["timestamp"], row["close"],
                        marker="v", color="#FF1744", s=60, zorder=5)

    last_price  = df_plot["close"].iloc[-1]
    last_signal = df_plot["signal"].iloc[-1]
    sig_color   = SIGNAL_COLORS.get(last_signal, "#CCCCCC")
    ax1.set_title(
        f"{SYMBOL} — Batch #{round_num} | "
        f"Price: ${last_price:,.2f} | "
        f"Signal: {last_signal}",
        color=sig_color, fontsize=12, fontweight="bold"
    )
    ax1.set_ylabel("Price (USDT)", color="#CCCCCC")
    ax1.legend(loc="upper left", facecolor="#2D2D44", labelcolor="#CCCCCC")
    ax1.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))

    # ── Panel 2: RSI ─────────────────────────────────────────
    ax2 = axes[1]
    rsi_values = df_plot["rsi"].dropna()
    rsi_ts     = df_plot.loc[rsi_values.index, "timestamp"]
    ax2.plot(rsi_ts, rsi_values, color="#CE93D8", linewidth=1.5, label="RSI")
    ax2.axhline(70, color="#FF4444", linestyle="--", alpha=0.7, linewidth=1)
    ax2.axhline(30, color="#00C851", linestyle="--", alpha=0.7, linewidth=1)
    ax2.axhline(50, color="#888888", linestyle=":",  alpha=0.5, linewidth=0.8)
    ax2.fill_between(rsi_ts, 70, 100, alpha=0.1, color="#FF4444")
    ax2.fill_between(rsi_ts,  0, 30,  alpha=0.1, color="#00C851")
    ax2.set_ylim(0, 100)
    ax2.set_ylabel("RSI", color="#CCCCCC")
    ax2.text(rsi_ts.iloc[-1], 72, "Overbought", color="#FF4444",
             fontsize=7, ha="right")
    ax2.text(rsi_ts.iloc[-1], 22, "Oversold",   color="#00C851",
             fontsize=7, ha="right")
    ax2.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    last_rsi = df_plot["rsi"].iloc[-1]
    ax2.set_title(f"RSI: {last_rsi:.1f}", color="#CE93D8", fontsize=10)

    # ── Panel 3: MACD Histogram ──────────────────────────────
    ax3 = axes[2]
    hist_values = df_plot["macd_hist"].dropna()
    hist_ts     = df_plot.loc[hist_values.index, "timestamp"]
    colors_hist = ["#00C851" if v >= 0 else "#FF4444" for v in hist_values]
    ax3.bar(hist_ts, hist_values, color=colors_hist, width=0.0005, alpha=0.8)
    ax3.plot(hist_ts, df_plot.loc[hist_values.index, "macd"],
             color="#4FC3F7", linewidth=1, label="MACD")
    ax3.plot(hist_ts, df_plot.loc[hist_values.index, "macd_signal"],
             color="#FFD54F", linewidth=1, label="Signal")
    ax3.axhline(0, color="#888888", linewidth=0.8)
    ax3.set_ylabel("MACD", color="#CCCCCC")
    ax3.legend(loc="upper left", facecolor="#2D2D44", labelcolor="#CCCCCC",
                fontsize=8)
    ax3.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    last_macd = df_plot["macd"].iloc[-1]
    ax3.set_title(f"MACD: {last_macd:.2f}", color="#4FC3F7", fontsize=10)

    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

    # ── Bảng dữ liệu 10 nến mới nhất ────────────────────────
    table_cols = ["timestamp", "close", "rsi", "macd",
                  "signal", "ml_lr", "ml_rf", "rf_proba"]
    table_df = df_plot[table_cols].tail(10).copy()
    table_df["close"]    = table_df["close"].map("${:,.2f}".format)
    table_df["rsi"]      = table_df["rsi"].map("{:.1f}".format)
    table_df["macd"]     = table_df["macd"].map("{:.2f}".format)
    table_df["rf_proba"] = table_df["rf_proba"].map("{:.2%}".format)
    print(f"\n📋 10 nến mới nhất (tổng {storage.count()} nến đã xử lý):")
    display(table_df.reset_index(drop=True))


print("✅ Hàm render_dashboard đã sẵn sàng.")

## Bước 12: Chạy Pipeline Streaming Real-time

Khởi động toàn bộ pipeline:
1. Producer thread liên tục fetch giá từ Binance API
2. Consumer (Spark) đọc từ queue, xử lý và dự đoán mỗi `STREAM_INTERVAL` giây
3. Kết quả được lưu vào CSV và hiển thị real-time

In [ ]:
# ─────────────────────────────────────────────────────────────
# Bước 12: Chạy Pipeline Streaming Real-time
#
# Pipeline:
#   Binance API
#     → CryptoProducer (threading)
#       → MockKafkaBroker (queue.Queue)
#         → SparkStreamProcessor (micro-batch)
#           → ResultStorage (CSV)
#             → render_dashboard (matplotlib)
# ─────────────────────────────────────────────────────────────

print("🚀 Khởi động pipeline streaming real-time...")
print(f"   Sẽ chạy {STREAM_ROUNDS} batch, mỗi batch cách {STREAM_INTERVAL}s")
print(f"   Topic: '{CryptoProducer.TOPIC}'")
print("-" * 60)

# Khởi động Producer (background thread)
producer.start()

# Xử lý từng batch
for batch_num in range(1, STREAM_ROUNDS + 1):
    start_time = time.time()

    # Consumer: poll từ Kafka queue
    messages = kafka_broker.poll(
        CryptoProducer.TOPIC, timeout=STREAM_INTERVAL - 0.5
    )

    queue_size = kafka_broker.qsize(CryptoProducer.TOPIC)
    print(f"\n⏱️  Batch #{batch_num}/{STREAM_ROUNDS} "
          f"| Messages: {len(messages)} "
          f"| Queue remaining: {queue_size}")

    # Nếu không có message mới, dùng dữ liệu cuối từ history
    if not messages:
        print("   ℹ️  Không có message mới, bỏ qua batch này.")
        # Vẫn render nếu có dữ liệu trong storage
        df_current = storage.read()
        if not df_current.empty:
            clear_output(wait=True)
            render_dashboard(df_current, batch_num)
        time.sleep(max(0, STREAM_INTERVAL - (time.time() - start_time)))
        continue

    # Spark micro-batch processing
    spark_result = processor.process_batch(messages)

    if spark_result is not None:
        # Lưu vào storage (CSV)
        storage.write(spark_result)
        print(f"   💾 Đã lưu {spark_result.count()} nến | "
              f"Tổng: {storage.count()} nến")

        # Hiển thị real-time
        df_display = storage.read()
        clear_output(wait=True)
        render_dashboard(df_display, batch_num)
    else:
        print("   ⚠️  Chưa đủ dữ liệu để tính chỉ báo.")

    elapsed = time.time() - start_time
    sleep_time = max(0, STREAM_INTERVAL - elapsed)
    if sleep_time > 0:
        time.sleep(sleep_time)

# Dừng Producer
producer.stop()
print("\n" + "=" * 60)
print(f"✅ Pipeline kết thúc! Đã xử lý {storage.count()} nến tổng.")
print(f"📁 Kết quả đã lưu tại: {OUTPUT_CSV}")

## Bước 13: Phân tích kết quả cuối cùng

Đọc CSV và phân tích tổng hợp kết quả dự đoán.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Bước 13: Phân tích kết quả cuối cùng
# ─────────────────────────────────────────────────────────────

# Đọc từ CSV (giả lập đọc từ database)
df_final = pd.read_csv(OUTPUT_CSV)
df_final["timestamp"] = pd.to_datetime(df_final["timestamp"])
df_final = df_final.sort_values("timestamp").reset_index(drop=True)

print("=" * 60)
print("📊 TỔNG HỢP KẾT QUẢ")
print("=" * 60)
print(f"   Tổng số nến    : {len(df_final)}")
print(f"   Thời gian bắt đầu : {df_final['timestamp'].min()}")
print(f"   Thời gian kết thúc: {df_final['timestamp'].max()}")
print(f"   Giá đầu : ${df_final['close'].iloc[0]:,.2f}")
print(f"   Giá cuối: ${df_final['close'].iloc[-1]:,.2f}")

price_change = (df_final['close'].iloc[-1] / df_final['close'].iloc[0] - 1) * 100
trend = "📈" if price_change > 0 else "📉"
print(f"   Biến động  : {trend} {price_change:+.2f}%")

print("\n--- Phân phối tín hiệu Rule-based ---")
print(df_final["signal"].value_counts().to_string())

print("\n--- Phân phối dự đoán ML (Logistic Regression) ---")
print(df_final["ml_lr"].value_counts().to_string())

print("\n--- Phân phối dự đoán ML (Random Forest) ---")
print(df_final["ml_rf"].value_counts().to_string())

print("\n--- Thống kê RSI ---")
rsi_valid = df_final["rsi"].dropna()
print(f"   Min: {rsi_valid.min():.1f} | Mean: {rsi_valid.mean():.1f} | Max: {rsi_valid.max():.1f}")
print(f"   Overbought (>70): {(rsi_valid > 70).sum()} nến")
print(f"   Oversold   (<30): {(rsi_valid < 30).sum()} nến")

print("\n--- 10 hàng cuối trong CSV ---")
display(df_final.tail(10))

In [ ]:
# ─────────────────────────────────────────────────────────────
# Biểu đồ tổng hợp cuối cùng với toàn bộ dữ liệu
# ─────────────────────────────────────────────────────────────

render_dashboard(df_final, round_num=STREAM_ROUNDS)

# ─── Biểu đồ phân phối tín hiệu ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor("#1E1E2E")

for ax in axes:
    ax.set_facecolor("#2D2D44")
    ax.tick_params(colors="#CCCCCC")
    ax.spines[:].set_color("#444466")

# Rule-based signal distribution
sig_counts = df_final["signal"].value_counts()
axes[0].bar(
    sig_counts.index,
    sig_counts.values,
    color=[SIGNAL_COLORS.get(s, "#888888") for s in sig_counts.index],
    alpha=0.8
)
axes[0].set_title("Rule-based Signal Distribution",
                   color="#CCCCCC", fontweight="bold")
axes[0].set_ylabel("Count", color="#CCCCCC")

# ML (Random Forest) signal distribution
rf_counts = df_final["ml_rf"].value_counts()
axes[1].bar(
    rf_counts.index,
    rf_counts.values,
    color=[SIGNAL_COLORS.get(s, "#888888") for s in rf_counts.index],
    alpha=0.8
)
axes[1].set_title("Random Forest Prediction Distribution",
                   color="#CCCCCC", fontweight="bold")
axes[1].set_ylabel("Count", color="#CCCCCC")

plt.suptitle(f"{SYMBOL} — Kết quả dự đoán ({len(df_final)} nến)",
             color="#4FC3F7", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f"\n🎉 Hoàn thành! File kết quả: {OUTPUT_CSV}")

## Bước 14: Spark SQL — Query kết quả

Dùng Spark SQL để truy vấn dữ liệu kết quả, giống như query từ Cassandra.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Bước 14: Spark SQL — Phân tích kết quả với Spark
# ─────────────────────────────────────────────────────────────

# Tải kết quả vào Spark DataFrame
spark_results = spark.read.csv(OUTPUT_CSV, header=True, inferSchema=True)
spark_results.createOrReplaceTempView("crypto_predictions")

print("✅ Đã tải kết quả vào Spark DataFrame.")
print(f"   Số hàng: {spark_results.count()}")
print("\nSchema:")
spark_results.printSchema()

# ─── Query 1: Tổng hợp theo tín hiệu ─────────────────────────
print("\n--- Query 1: Phân phối tín hiệu và giá trung bình ---")
spark.sql("""
    SELECT
        signal,
        COUNT(*)                    AS count,
        ROUND(AVG(close),  2)       AS avg_price,
        ROUND(AVG(rsi),    2)       AS avg_rsi,
        ROUND(AVG(rf_proba) * 100, 2) AS avg_rf_proba_pct
    FROM crypto_predictions
    GROUP BY signal
    ORDER BY count DESC
""").show()

# ─── Query 2: RSI phân vùng ───────────────────────────────────
print("--- Query 2: Phân vùng RSI ---")
spark.sql("""
    SELECT
        CASE
            WHEN rsi > 70 THEN 'Overbought (>70)'
            WHEN rsi < 30 THEN 'Oversold (<30)'
            ELSE               'Neutral (30-70)'
        END AS rsi_zone,
        COUNT(*) AS count,
        ROUND(AVG(close), 2) AS avg_price
    FROM crypto_predictions
    WHERE rsi IS NOT NULL
    GROUP BY rsi_zone
    ORDER BY count DESC
""").show()

# ─── Query 3: Thống kê giá ───────────────────────────────────
print("--- Query 3: Thống kê giá tổng hợp ---")
spark.sql("""
    SELECT
        symbol,
        COUNT(*)                AS total_candles,
        ROUND(MIN(close), 2)    AS min_price,
        ROUND(MAX(close), 2)    AS max_price,
        ROUND(AVG(close), 2)    AS avg_price,
        ROUND(STDDEV(close), 2) AS price_stddev,
        ROUND(AVG(rsi), 2)      AS avg_rsi,
        ROUND(AVG(macd), 4)     AS avg_macd
    FROM crypto_predictions
    GROUP BY symbol
""").show()

print("✅ Spark SQL analysis hoàn thành!")

## Bước 15: Dọn dẹp

Dừng SparkSession sau khi hoàn thành.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Bước 15: Dọn dẹp
# ─────────────────────────────────────────────────────────────

spark.stop()
print("✅ SparkSession đã dừng.")
print("\n" + "=" * 60)
print("🎉 ĐỒ ÁN HOÀN CHỈNH!")
print("=" * 60)
print("""
Pipeline đã thực hiện:
  1. ✅ Cài đặt môi trường (PySpark, scikit-learn, ...)
  2. ✅ Lấy dữ liệu từ Binance API
  3. ✅ Kafka Producer giả lập (queue.Queue + threading)
  4. ✅ Spark Streaming Consumer (micro-batch)
  5. ✅ Feature Engineering: RSI + MACD
  6. ✅ Rule-based Prediction (RSI/MACD rules)
  7. ✅ ML Prediction (Logistic Regression + Random Forest)
  8. ✅ Lưu kết quả vào CSV (giả lập Cassandra/HBase)
  9. ✅ Real-time Visualization (matplotlib)
 10. ✅ Spark SQL analysis
""")